# Verify: Whisper-Hindi2Hinglish streaming swap (Kaggle GPU)

Confirms the new Hinglish model **loads**, **produces faithful Hinglish text**, and the
streaming `draft()` **end-to-final** works, with per-clip latency.

**Before running** &rarr; Notebook **Settings**:
- **Accelerator: GPU T4 x2** (the script pins itself to one GPU, `cuda:0`)
- **Internet: ON** (needed to clone the repo + download the model from Hugging Face)

Then **Run All**. CUDA here is a stand-in for the M1's MPS — the transformers load path is
device-independent, so a clean result means it's ready to send to the owner.

In [ ]:
# 1) Environment / GPU check
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| cuda?", torch.cuda.is_available(),
      "| n_gpu", torch.cuda.device_count())

In [ ]:
# 2) Clone the repo and install the (Whisper) dependencies
!rm -rf repo && git clone -q https://github.com/arnav-chauhan-kgpian/hindi-english-raaaa.git repo
%cd repo
!pip -q install -r requirements.txt
import transformers, faster_whisper  # noqa: F401
print("transformers", transformers.__version__)
# import sanity: the loader uses Auto* if present, else the concrete Whisper classes
try:
    from transformers import AutoModelForSpeechSeq2Seq; print('AutoModelForSpeechSeq2Seq: OK')
except Exception as e:
    print('AutoModelForSpeechSeq2Seq MISSING ->', e)
from transformers import WhisperForConditionalGeneration; print('WhisperForConditionalGeneration: OK')

In [ ]:
# 3) Run the verifier on ONE GPU (cuda:0). First run downloads the model (~6 GB) once.
#    Prints, per sample: direct Hinglish transcription (fidelity) + streaming end-to-final.
!CUDA_VISIBLE_DEVICES=0 python kaggle_verify_streaming.py

In [ ]:
# 4) (optional) Try the smaller/faster variant for a latency comparison
#    Apex ~800M is faster than Prime (large-v3); useful if Prime is too slow on the M1.
!CUDA_VISIBLE_DEVICES=0 STT_HINGLISH_MODEL=Oriserve/Whisper-Hindi2Hinglish-Apex python kaggle_verify_streaming.py

## What to look for
- **Text** is readable romanized Hinglish that keeps the Hindi-English mix (e.g. `mujhe kal AWS ka deployment dekhna hai`), not pure English or gibberish.
- **`hinglish loaded: True`** with no error — proves the model loads (the step that failed for qwen3-asr, and the `AutoModelForSpeechSeq2Seq` import issue we just fixed).
- **end-to-final latency** is low (sub-second on T4). On the M1's MPS expect roughly 1–2× this.

Paste the full output back and I'll confirm it's good to send to the owner (or pick the variant that best balances fidelity vs. latency).